In [0]:
CREATE OR REPLACE VIEW gold.vw_purchase_kpis AS

WITH monthly_purchase AS (

    SELECT
        DATE_TRUNC('month', purchase_date) AS month,

        COUNT(DISTINCT purchase_order_id) AS total_orders,

        ROUND(
            SUM(total_amount),
            2
        ) AS total_spend,

        COUNT(DISTINCT supplier_key) AS active_suppliers

    FROM gold.fact_purchases

    WHERE purchase_date IS NOT NULL
      AND purchase_status IN ('purchase', 'done')

    GROUP BY DATE_TRUNC('month', purchase_date)

),

supplier_first_purchase AS (

    SELECT
        supplier_key,

        MIN(
            DATE_TRUNC('month', purchase_date)
        ) AS first_purchase_month

    FROM gold.fact_purchases

    WHERE purchase_status IN ('purchase', 'done')

    GROUP BY supplier_key

),

new_suppliers AS (

    SELECT
        first_purchase_month AS month,

        COUNT(DISTINCT supplier_key) AS new_suppliers

    FROM supplier_first_purchase

    GROUP BY first_purchase_month

),

pending_approvals AS (

    SELECT
        COUNT(DISTINCT purchase_order_id)
            AS pending_approvals

    FROM gold.fact_purchases

    WHERE purchase_status IN (
        'draft',
        'sent',
        'to approve'
    )

)

SELECT
    mp.month,

    mp.total_orders,

    LAG(mp.total_orders)
        OVER (ORDER BY mp.month)
            AS prev_month_orders,

    ROUND(
        (
            mp.total_orders -
            LAG(mp.total_orders)
                OVER (ORDER BY mp.month)
        )
        /
        NULLIF(
            LAG(mp.total_orders)
                OVER (ORDER BY mp.month),
            0
        ) * 100,
        2
    ) AS orders_growth_percent,

    mp.total_spend,

    LAG(mp.total_spend)
        OVER (ORDER BY mp.month)
            AS prev_month_spend,

    ROUND(
        (
            mp.total_spend -
            LAG(mp.total_spend)
                OVER (ORDER BY mp.month)
        )
        /
        NULLIF(
            LAG(mp.total_spend)
                OVER (ORDER BY mp.month),
            0
        ) * 100,
        2
    ) AS spend_growth_percent,

    mp.active_suppliers,

    COALESCE(
        ns.new_suppliers,
        0
    ) AS new_suppliers_this_month,

    pa.pending_approvals

FROM monthly_purchase mp

LEFT JOIN new_suppliers ns
    ON mp.month = ns.month

CROSS JOIN pending_approvals pa;

In [0]:
Select * from smart_odoo.bronze.purchase_order